# 10 — Análisis y primera gráfica

Pregunta: ¿qué segmento concentra más saldo aceptado en un corte? Usamos 48 registros ficticios; la gráfica describe solo los que pasan las reglas. Lee el README y el contrato antes de ejecutar. Reinicia el kernel y ejecuta todas las celdas en orden.


## 1. Entorno y rutas

Selecciona un kernel con pandas y Matplotlib. Esta celda localiza el proyecto desde la raíz o una carpeta interna y permite importar analysis.py. Es preparación del ejemplo; no modifica archivos de configuración.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

relativa = Path("curriculum/02_pandas/10_analysis_plot")
candidatas = [Path.cwd(), *Path.cwd().parents]
raiz = next((p for p in candidatas if (p / relativa / "analysis.py").is_file()), None)
if raiz is None:
    raise RuntimeError("Abre el notebook dentro de la carpeta ds_track")
carpeta = raiz / relativa
if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))
from analysis import leer_fuente, limpiar_y_separar, resumir_corte, crear_grafica, redactar_hallazgo, exportar_tablas
print("pandas:", pd.__version__, "Matplotlib:", matplotlib.__version__)


## 2. Declarar la pregunta y cargar

Modifica corte para otra pregunta. No cambies los datos para obtener un resultado deseado. Las salidas se regeneran con el corte elegido.


In [ ]:
corte = "2026-10-31"
entrada = carpeta.parent / "06_text_dates" / "cartera_texto_fechas.csv"
fuente = leer_fuente(entrada)
original = fuente.copy()
print("Registros de entrada:", len(fuente))
fuente.head()


## 3. Reglas y registros apartados

Se conservan originales y todos los motivos. La limpieza no inventa categorías ni fechas. La política está documentada en el README.


In [ ]:
aceptados, rechazados = limpiar_y_separar(fuente)
assert len(aceptados) + len(rechazados) == len(fuente)
print("Aceptados:", len(aceptados), "Rechazados:", len(rechazados))
rechazados[["numero_registro", "id_credito_original", "motivos"]]


## 4. Delimitar el corte y conciliar

El denominador de participación es el saldo aceptado del corte. Los créditos se cuentan dentro de esa fecha, no sumando los dos cortes.


In [ ]:
seleccion, resumen = resumir_corte(aceptados, corte)
print("Créditos del corte:", len(seleccion))
print("Saldo aceptado:", seleccion["saldo_mxn"].sum(min_count=1))
resumen


## 5. Dibujar desde la tabla comprobada

Las barras comienzan en cero, tienen unidades y usan el mismo resumen. Modificar el estilo no modifica los datos. No se crean barras ficticias cuando el corte no tiene observaciones.


In [ ]:
fig, ax = crear_grafica(resumen, corte, len(seleccion))
plt.show()


## 6. Explicar el resultado

Lee el borrador de hallazgo y escribe debajo tu interpretación: pregunta, resultado, evidencia y límites. Una barra mayor no explica la causa ni demuestra un promedio mayor.


In [ ]:
hallazgo = redactar_hallazgo(resumen, corte, aceptados, rechazados)
print(hallazgo)


### Mi interpretación

Sustituye este texto por tu explicación. ¿Qué muestra la figura y qué necesitarías para comparar poblaciones o estudiar una causa?


## 7. Exportar y comprobar la fuente

La copia de práctica queda en output/notebook, ignorada por Git. Puede reemplazarse al repetir esta celda. Al entregar, reinicia y ejecuta todo; elimina salidas si muestran rutas personales.


In [ ]:
salida = carpeta / "output" / "notebook"
exportar_tablas({"aceptados.csv": aceptados, "rechazados.csv": rechazados,
                "detalle_corte.csv": seleccion, "resumen_corte.csv": resumen}, entrada, salida)
fig.savefig(salida / "saldo_segmentos.png", dpi=160)
(salida / "hallazgo.md").write_text(hallazgo, encoding="utf-8")
pd.testing.assert_frame_equal(fuente, original)
plt.close(fig)
print("Exportaciones comprobadas; fuente intacta")


## Continuidad

Realiza el [lab 02.10](../../../labs/02_pandas/10_analysis_plot/README.md). El módulo tiene material disponible, pero cerrarlo requiere tu evidencia. Después continúa con [Modelo relacional y SQLite](../../03_sql_databases/01_relational_sqlite/README.md).
